# Geometry 02 — Prouver par l'algèbre

**Public : Licence** — deuxième étape de la série [Geometry](README.md), le programme gradué de la preuve automatique en géométrie.

**Ce que ce notebook suppose.** Le notebook [Geometry-01 — De la figure à l'équation](Geometry-01-From-Figure-To-Equation.ipynb) : traduire une figure en hypothèse polynomiale $H$, une propriété en conclusion $C$, et tester $C$ sous $H$ par tirages aléatoires (Schwartz-Zippel).

**Ce que vous emportez.** Le passage du *test* à la *preuve* : l'idéal engendré par les hypothèses, l'appartenance de la conclusion à cet idéal via une base de Gröbner, le certificat qui l'accompagne, et le phénomène central que le 01 avait laissé dans l'ombre — les **non-dégénérescences**, et leur traitement algébrique par saturation.

## Où en sommes-nous ?

Le 01 s'est arrêté sur un constat mesuré : sur $100$ figures rectangle tirées au hasard, la conclusion $C_1$ (le milieu de l'hypoténuse est équidistant de $A$ et $B$) s'est annulée $100$ fois, et Schwartz-Zippel bornait la probabilité d'un faux accord à $7{,}7 \times 10^{-171}$. Autant dire : aucun doute pratique.

Mais un test, même vert à $10^{-171}$ près, reste un test. La question que ce notebook pose — et résout — est d'une autre nature :

> $C_1$ s'annule-t-elle sur **toute** figure satisfaisant $H$, sans exception, sans probabilité ?

La réponse tiendra en une identité d'une ligne, $C_1 = 1 \cdot H$, que nous dégagerons par la machinerie générale des idéaux et des bases de Gröbner. Et cette machinerie nous réserve deux surprises : elle dit **mieux** que ce qu'on lui demande (le cercle circonscrit), elle sait dire **non** (le rectangle n'implique pas l'isocélé), et elle révèle que certains théorèmes géométriques sont **faux** tels qu'énoncés — jusqu'à ce qu'on leur adjoint des non-dégénérescences, dont l'algèbre a un traitement exact.

## Plan

1. **§1** Les objets du fil rouge, reconstruits
2. **§2** L'idéal engendré par les hypothèses : payer la conclusion avec les hypothèses
3. **§3** Bases de Gröbner : l'appartenance décidable
4. **§4** Le certificat dit mieux : le cercle circonscrit
5. **§5** La machine dit non : le rectangle n'implique pas l'isocèle
6. **§6** Deux hypothèses : le vrai besoin, et un théorème faux
7. **§7** L'astuce de Ritt : rendre une non-dégénérescence inversible
8. **§8** Le prix : coût de la certitude
9. **§9-§11** Trois exercices
10. **§12** La décision réelle (Z3) : un autre organe, une même question — et le discriminant réels/complexes

In [1]:
import sys
import time

import sympy as sp

print("Python :", sys.version.split()[0])
print("sympy  :", sp.__version__)

Python : 3.13.15
sympy  : 1.14.0


### Lecture du résultat

La sortie confirme l'environnement d'exécution : **Python 3.13.15**, **sympy 1.14.0**. Tout ce notebook est déterministe — aucun tirage aléatoire — donc ces versions sont aussi celles qui ont produit les sorties commitées ci-dessous : les réexécuter avec les mêmes versions redonne exactement les mêmes polynômes et les mêmes verdicts.

### §1 — Les objets du fil rouge

Exactement comme au 01 : trois points $A$, $B$, $C$ à coordonnées symboliques, l'hypothèse « angle droit en $A$ » comme nullité du produit scalaire $\vec{AB} \cdot \vec{AC}$, et la conclusion comme différence de carrés de distances au milieu $M$ de $[B, C]$.

$$H = (x_B - x_A)(x_C - x_A) + (y_B - y_A)(y_C - y_A) \qquad C_1 = MA^2 - MB^2$$

Tout le notebook est **déterministe** : aucun tirage aléatoire, aucune probabilité. C'est la marque du changement de nature — on ne teste plus, on démontre.
La cellule suivante reconstruit ces objets et imprime $H$ et $C_1$ sous forme développée : c'est cette sortie, citée telle quelle, que toutes les réductions ultérieures prendront pour référence.

In [2]:
# Les objets du fil rouge, reconstruits comme au 01
xA, yA, xB, yB, xC, yC = sp.symbols('x_A y_A x_B y_B x_C y_C')

# Hypothese : angle droit en A, c'est-a-dire AB scalaire AC nul
H = (xB - xA) * (xC - xA) + (yB - yA) * (yC - yA)

# Milieu M de l'hypotenuse [B, C]
Mx, My = (xB + xC) / 2, (yB + yC) / 2

# Conclusion C1 : M equidistant de A et B (difference des carres)
C1 = sp.expand((Mx - xA) ** 2 + (My - yA) ** 2 - ((Mx - xB) ** 2 + (My - yB) ** 2))

print("H  =", H)
print("C1 =", C1)

H  = (-x_A + x_B)*(-x_A + x_C) + (-y_A + y_B)*(-y_A + y_C)
C1 = x_A**2 - x_A*x_B - x_A*x_C + x_B*x_C + y_A**2 - y_A*y_B - y_A*y_C + y_B*y_C


### Lecture du résultat

L'hypothèse $H$ est un polynôme à six variables ; la conclusion $C_1$ aussi, une fois le milieu substitué — la sortie affiche les deux formes développées, `H = (-x_A + x_B)*(-x_A + x_C) + (-y_A + y_B)*(-y_A + y_C)` et `C1 = x_A**2 - x_A*x_B - x_A*x_C + x_B*x_C + y_A**2 - y_A*y_B - y_A*y_C + y_B*y_C` : deux polynômes de degré 2 en six variables. La question « toute figure rectangle satisfait-elle $C_1 = 0$ ? » devient :

> le polynôme $C_1$ s'annule-t-il **partout où** le polynôme $H$ s'annule ?

### §2 — L'idéal engendré par les hypothèses

L'observation clé est algébrique. Considérons l'ensemble de tous les multiples de $H$ :

$$\langle H \rangle = \{ A \cdot H \;:\; A \ \text{polynôme} \}$$

C'est l'**idéal** engendré par $H$. Il a une propriété de transfert immédiate :

> si $C \in \langle H \rangle$, alors $C$ s'annule partout où $H$ s'annule.

La raison tient en une ligne : si $C = A \cdot H$ et $H(\text{figure}) = 0$, alors $C(\text{figure}) = A(\text{figure}) \cdot 0 = 0$. L'écriture $C = A \cdot H$ est un **certificat** : quiconque peut le vérifier d'un simple développement. On dit que la conclusion est *payée cash* par les hypothèses.

Notre idéal est *principal* (un seul générateur) — le cas le plus simple possible. Le premier geste, naturel, est de tenter la division de $C_1$ par $H$.

In [3]:
# Division de C1 par H : quotient et reste disent tout
quotient, reste = sp.div(C1, H, xA, yA, xB, yB, xC, yC)
print("quotient :", quotient)
print("reste    :", reste)

# Verification independante du certificat : C1 - quotient * H doit etre nul
print("certificat C1 = quotient * H :", sp.expand(C1 - quotient * H) == 0)

quotient : 1
reste    : 0
certificat C1 = quotient * H : True


### Lecture du résultat — le théorème est démontré

$q = 1$, $r = 0$ : la division ne laisse **aucun reste**, et le certificat se vérifie par un simple développement — la sortie l'imprime telle quelle : `quotient : 1`, `reste : 0`, `certificat C1 = quotient * H : True`.

$$C_1 = 1 \cdot H$$

Autrement dit : après substitution du milieu, *la conclusion est exactement l'hypothèse*. Le théorème du milieu de l'hypoténuse tient en une identité polynomiale. Sa portée dépasse tout ce que le 01 pouvait affirmer :

- **toute** figure, pas une famille paramétrée ;
- y compris les figures **dégénérées** ($A = B$, $A = C$, points confondus) ;
- sur tout corps de coefficients, réels ou complexes.

Le 01 mesurait une probabilité ; le 02 exhibe un certificat. La probabilité disparaît de l'énoncé.

### §3 — Bases de Gröbner : l'appartenance décidable

Si $\langle H \rangle$ était toujours principal, une division suffirait et ce notebook s'arrêterait ici. Mais dès qu'un théorème a **deux** hypothèses $H_1, H_2$, l'idéal $\langle H_1, H_2 \rangle$ contient les combinaisons $A \cdot H_1 + B \cdot H_2$, et la division naïve par une liste de polynômes dépend de l'**ordre** des diviseurs et de l'ordre des monômes : le même calcul peut laisser un reste non nul alors que la conclusion appartient à l'idéal (l'exemple canonique $f = xy^2 - x$ divisé par $xy + 1$ et $y - 1$ est détaillé au chapitre 2 de Cox, Little et O'Shea, *Ideals, Varieties, and Algorithms*).

La **base de Gröbner** est la réparation exacte : une base de l'idéal dont le reste de division est **canonique** — indépendant de l'ordre des diviseurs — ce qui rend l'appartenance décidable :

> $C \in \langle H_1, \ldots, H_s \rangle$ si et seulement si le reste de $C$ par la base de Gröbner est nul.

Et le calcul rend les **coefficients** du certificat $C = \sum_i A_i G_i$. Appliquons la machinerie générale à notre idéal — trivial ici, mais c'est elle qui portera les sections suivantes.

In [4]:
# Base de Groebner de l'ideal <H>, ordre lexicographique
G = sp.groebner([H], xA, yA, xB, yB, xC, yC, order='lex')
print("G =", G)

coefs, reste = G.reduce(C1)
print("coefficients :", coefs)
print("reste        :", reste)
print("certificat   :", sp.expand(coefs[0] * H - C1) == 0)

G = GroebnerBasis([x_A**2 - x_A*x_B - x_A*x_C + x_B*x_C + y_A**2 - y_A*y_B - y_A*y_C + y_B*y_C], x_A, y_A, x_B, y_B, x_C, y_C, domain='ZZ', order='lex')
coefficients : [1]
reste        : 0
certificat   : True


### Lecture du résultat

Pour un idéal principal, la base de Gröbner est le générateur lui-même, réécrit tête d'abord — la sortie le confirme : `G = GroebnerBasis([x_A**2 - x_A*x_B - x_A*x_C + x_B*x_C + y_A**2 - y_A*y_B - y_A*y_C + y_B*y_C], x_A, y_A, x_B, y_B, x_C, y_C, domain='ZZ', order='lex')`, puis `coefficients : [1]`, `reste : 0`, `certificat : True`. Aucune surprise. Ce qui compte est le **verdict** : reste $0$, coefficients $[1]$. La machinerie générale confirme l'appartenance de $C_1$ et redis le certificat du §2.

À partir de maintenant, nous disposons d'un **décideur d'appartenance** : pour n'importe quelle conclusion $C$, `G.reduce(C)` répond *oui* (reste nul, avec certificat) ou *non* (reste non nul). Les trois sections qui suivent le mettent à l'épreuve dans les trois régimes possibles : il en dit plus, il dit non, il échoue — et l'échec est informatif.

### §4 — Le certificat dit mieux : le cercle circonscrit

Le 01 concluait « $MA = MB$ ». Mais un théorème de géométrie en cache souvent d'autres. Deux questions naturelles :

- $C_p = MA^2 - MC^2$ (équidistance de $A$ et $C$) : est-elle elle aussi payée par les hypothèses ?
- $C_3 = MB^2 - MC^2$ (équidistance de $B$ et $C$) : que dit la machine ?
La cellule suivante mesure : elle affiche les coefficients et le reste de $C_p$ et de $C_3$ par $G$, tels quels.

In [5]:
# Cp : equidistance de A et C ; C3 : equidistance de B et C
Cp = sp.expand((Mx - xA) ** 2 + (My - yA) ** 2 - ((Mx - xC) ** 2 + (My - yC) ** 2))
C3 = sp.expand((Mx - xB) ** 2 + (My - yB) ** 2 - ((Mx - xC) ** 2 + (My - yC) ** 2))

coefs_p, reste_p = G.reduce(Cp)
print("Cp : coefficients =", coefs_p, "| reste =", reste_p)
print("C3 identiquement nul :", C3 == 0, "| reste par G =", G.reduce(C3)[1])

Cp : coefficients = [1] | reste = 0
C3 identiquement nul : True | reste par G = 0


### Lecture du résultat — trois équidistances

Trois régimes distincts, mesurés :

- $C_1 \in \langle H \rangle$ avec coefficient $1$ : payée par l'hypothèse ;
- $C_p \in \langle H \rangle$ avec coefficient $1$ également — c'est encore $C_p = H$ : payée aussi ;
- $C_3$ est **identiquement nul** : $M$ est le milieu de $[B, C]$, donc $MB = MC$ est vrai de *tout* triangle, rectangle ou non — aucune hypothèse nécessaire.

En composant : $MA = MB$, $MA = MC$, $MB = MC$ — le point $M$ est équidistant des **trois** sommets. Le triangle rectangle a un cercle circonscrit, et son centre est le milieu de l'hypoténuse. Le théorème complet du 01 était plus riche que sa conclusion : la machine l'a dit sans qu'on le demande.

Retenons aussi la distinction fine entre deux façons d'être « toujours vrai » : par **identité** ($C_3$, vrai de tout triangle) ou par **appartenance à l'idéal** ($C_1$, vrai de tout triangle *rectangle*).

### §5 — La machine dit non : le rectangle n'implique pas l'isocélé

Décidons l'appartenance d'une conclusion **fausse**. Candidat : « tout triangle rectangle est isocèle », c'est-à-dire $C_{iso} = AB^2 - AC^2$.
La cellule suivante produit deux verdicts qui devront s'accorder : le reste de $C_{iso}$ par $G$ (décision algébrique), puis l'évaluation de $H$ et $C_{iso}$ sur le triangle $3$-$4$-$5$ (décision numérique).

In [6]:
# Temoin negatif : le rectangle implique-t-il AB = AC ?
Ciso = (xB - xA) ** 2 + (yB - yA) ** 2 - ((xC - xA) ** 2 + (yC - yA) ** 2)
reste_iso = G.reduce(Ciso)[1]
print("reste de Ciso par G :", reste_iso)
print("reste nul ?", reste_iso == 0)

# Contre-exemple : u = (3, 0), v = (0, 4) -- triangle 3-4-5, rectangle non isocele
sub_345 = {xA: 0, yA: 0, xB: 3, yB: 0, xC: 0, yC: 4}
print("sur la figure 3-4-5 : H =", H.subs(sub_345), "| Ciso =", Ciso.subs(sub_345))

reste de Ciso par G : -2*x_A*x_B + 2*x_A*x_C + x_B**2 - x_C**2 - 2*y_A*y_B + 2*y_A*y_C + y_B**2 - y_C**2
reste nul ? False
sur la figure 3-4-5 : H = 0 | Ciso = -7


### Lecture du résultat

Le reste est **non nul** : $C_{iso} \notin \langle H \rangle$. Parce que le reste par une base de Gröbner est canonique, sa non-nullité est une **preuve** de non-appartenance — la machine ne se trompe pas de sens.

Et le terrain confirme : sur le triangle $3$-$4$-$5$, $H = 0$ mais $C_{iso} = -7$ ($AB = 3$, $AC = 4$). Le « théorème » est faux, et l'algèbre l'avait dit avant le contre-exemple.

Une nuance honnête pour aller au fond : l'appartenance à l'idéal **implique** l'annulation sur les figures, mais la réciproque est plus subtile — sur les complexes, c'est l'appartenance à la *racine* de l'idéal qui caractérise l'annulation (Nullstellensatz de Hilbert). Pour trancher qu'une conclusion n'est *pas* une conséquence géométrique, le contre-exemple explicite reste l'argument décisif ; la machine non nulle + le $3$-$4$-$5$ font ici cause commune.

### §6 — Deux hypothèses : le vrai besoin, et un théorème faux

Passons à un théorème à **deux** hypothèses — le régime où la base de Gröbner devient nécessaire. Énoncé candidat :

> si $AB \perp AC$ et $AB \perp AD$, alors $C$, $A$, $D$ sont alignés.

(Deux perpendiculaires à une même droite en un même point : même direction.) Pour alléger, nous fixons $A$ à l'origine par translation — un geste de normalisation standard, la propriété étant invariante par isométrie. Les hypothèses et la conclusion deviennent :

$$H_{2a} = x_B x_C + y_B y_C \qquad H_{2b} = x_B x_D + y_B y_D \qquad C = x_C y_D - x_D y_C$$

La conclusion est le produit vectoriel $\vec{AC} \wedge \vec{AD}$ : nul si et seulement si les vecteurs sont alignés.

In [7]:
# A fixe a l'origine : six inconnues restantes
xD, yD = sp.symbols('x_D y_D')
H2a = xB * xC + yB * yC    # AB scalaire AC = 0
H2b = xB * xD + yB * yD    # AB scalaire AD = 0
CONCL = xC * yD - xD * yC  # A, C, D alignes (produit vectoriel nul)
print("H2a   =", H2a)
print("H2b   =", H2b)
print("CONCL =", CONCL)

G3 = sp.groebner([H2a, H2b], xB, yB, xC, yC, xD, yD, order='lex')
print("G3 =", G3)
reste3 = G3.reduce(CONCL)[1]
print("reste de CONCL :", reste3, "| nul ?", reste3 == 0)

H2a   = x_B*x_C + y_B*y_C
H2b   = x_B*x_D + y_B*y_D
CONCL = x_C*y_D - x_D*y_C
G3 = GroebnerBasis([x_B*x_C + y_B*y_C, x_B*x_D + y_B*y_D, x_C*y_B*y_D - x_D*y_B*y_C], x_B, y_B, x_C, y_C, x_D, y_D, domain='ZZ', order='lex')
reste de CONCL : x_C*y_D - x_D*y_C | nul ? False


### Lecture du résultat — l'échec est informatif

Le reste est non nul : la machine **ne démontre pas** l'alignement à partir des seules hypothèses. Mais lisez la base :

$$G_3 = [\, x_B x_C + y_B y_C, \;\; x_B x_D + y_B y_D, \;\; y_B \cdot (x_C y_D - x_D y_C) \,]$$

Le troisième élément contient **la conclusion elle-même, multipliée par $y_B$**. Tout est en place dans l'idéal — sauf l'inversibilité du facteur $y_B$. Le théorème est-il pour autant faux ? Testons le cas qui fait douter.

> **Pour aller plus loin — Surviving proofs, *The Importance of Understanding* (05/09/2026)**
> Sheydvasser recommande, en principe 2, de *situer un échec dans son réseau de concepts*. Ici, la base de Gröbner $G_3$ ne démontre pas l'alignement à partir des seules hypothèses, mais elle exhibe **la conclusion elle-même** dans la base — c'est exactement l'inverse d'un mur silencieux : l'échec *dit quelque chose*. C'est le geste « lire structurellement » qu'elle défend à l'article 1 : un polynôme dans une base de Gröbner n'est pas juste un terme technique, c'est une position dans un réseau qui révèle où la preuve *manque*.
> *Senia Sheydvasser, The Deranged Mathematician (Substack), URL https://derangedmathematician.substack.com/p/the-importance-of-understanding*
> *Archivé : G:\Mon Drive\MyIA\IA\Bibliographie IA\Pedagogie-Proves\Sheydvasser-Surviving-Proofs\2026-09-05_the-importance-of-understanding.html*

In [8]:
# Contre-exemple : B = A rend les deux hypotheses trivialement vraies
sub_BA = {xB: 0, yB: 0, xC: 1, yC: 0, xD: 0, yD: 1}
print("H2a =", H2a.subs(sub_BA), "| H2b =", H2b.subs(sub_BA), "| CONCL =", CONCL.subs(sub_BA))

H2a = 0 | H2b = 0 | CONCL = 1


### Lecture du résultat — le théorème est faux tel qu'énoncé

Avec $B = A$ : les deux hypothèses valent $0$ *trivialement* (le vecteur $\vec{AB}$ est nul, « perpendiculaire » à tout), et la conclusion vaut $1$ — $C = (1,0)$ et $D = (0,1)$ ne sont pas alignés avec $A$.

Le théorème géométrique, énoncé avec des mots, sous-entendait **$B \neq A$** — la phrase « la perpendiculaire en $A$ à la droite $AB$ » n'a de sens que si $B \neq A$. Cette condition invisible dans les équations est une **non-dégénérescence**. Toute la géométrie automatisée vit avec elles : un énoncé honnête est un système d'hypothèses *plus* ses non-dégénérescences. La question devient : comment dire « $g \neq 0$ » dans un monde qui ne parle que de nullités de polynômes ?

> **Pour aller plus loin — Surviving proofs, *Counterexamples and Contradictions* (12/09/2026)**
> Sheydvasser recommande de chercher le contre-exemple le plus simple d'abord. Ici, $B = A$ est exactement ce contre-exemple *catégorique* : il rend les deux hypothèses trivialement vraies (le vecteur $\vec{AB}$ est nul, donc perpendiculaire à tout) et la conclusion trivialement fausse. C'est le pattern Sheydvasser art. 2 : « dégénéré par construction → généraliser l'énoncé pour exclure la dégénérescence » — ici en exigeant $B \neq A$, ce que la cellule [25] fait explicitement par saturation de l'idéal.
> *Senia Sheydvasser, The Deranged Mathematician (Substack), URL https://derangedmathematician.substack.com/p/counterexamples-and-contradictions*
> *Archivé : G:\Mon Drive\MyIA\IA\Bibliographie IA\Pedagogie-Proves\Sheydvasser-Surviving-Proofs\2026-09-12_counterexamples-and-contradictions.html*

### §7 — L'astuce de Ritt : rendre $g$ inversible

L'astuce (due à Ritt, systématisée par Wu) est d'une élégance totale : pour imposer $g \neq 0$, on adjoint à l'idéal le polynôme

$$z \cdot g - 1$$

où $z$ est une **nouvelle variable**. Si une figure satisfait les hypothèses et $z \, g = 1$, alors $g$ y est inversible — en particulier non nul. On démontre la conclusion dans l'idéal **saturé** ; l'opération s'appelle la saturation.

Ici $g = x_B^2 + y_B^2$ : sur les figures réelles, $g \neq 0$ signifie exactement $B \neq A$.
\nLa cellule suivante applique le geste : elle ajoute $z \cdot g - 1$ à la liste des générateurs et réduit `CONCL` — le reste affiché dira si le théorème tient sous $B \neq A$.

In [9]:
# Non-degenerescence B != A : rendre g = xB^2 + yB^2 inversible
z = sp.symbols('z')
g_nd = xB ** 2 + yB ** 2
G4 = sp.groebner([H2a, H2b, z * g_nd - 1], xB, yB, xC, yC, xD, yD, z, order='lex')
reste4 = G4.reduce(CONCL)[1]
print("reste de CONCL apres saturation :", reste4)
print("theoreme demontre sous B != A :", reste4 == 0)

reste de CONCL apres saturation : 0
theoreme demontre sous B != A : True


### Lecture du résultat — démontré, proprement cette fois

Reste nul : pour **toute** figure avec $B \neq A$, deux perpendiculaires à $AB$ en $A$ portent des points alignés avec $A$. Le théorème est maintenant vrai — c'est-à-dire : l'énoncé corrigé par sa non-dégénérescence est un théorème.

C'est exactement le geste que la méthode de Wu (objet du 03) industrialise : énoncé, hypothèses, conclusion, non-dégénérescences découvertes par la machinerie elle-même. Notons en passant une subtilité que nous laisserons au 03 : sur les *complexes*, $x_B^2 + y_B^2 = 0$ a d'autres solutions que $B = A$ (par exemple $x_B = 1$, $y_B = i$) — la géométrie *réelle* mérite son propre traitement, et la saturation exacte de $B \neq A$ réel demande un peu plus de soin. L'esprit du geste est intact.

### §8 — Le prix : coût de la certitude

La base de Gröbner est l'outil décisionnel central de l'algèbre commutative effective — au prix d'une complexité au pire cas **doublement exponentielle** en le nombre de variables : les degrés des éléments de la base peuvent exploser. Nos exemples sont minuscules ; mesurons-les quand même, pour fixer les ordres de grandeur.
La cellule suivante chronomètre les trois calculs — une hypothèse, deux hypothèses, deux hypothèses saturées — et affiche les trois temps en secondes.

In [10]:
# Trois echelles : 1 hypothese, 2 hypotheses, 2 hypotheses + saturation
t0 = time.perf_counter()
_ = sp.groebner([H], xA, yA, xB, yB, xC, yC, order='lex')
t1 = time.perf_counter()
_ = sp.groebner([H2a, H2b], xB, yB, xC, yC, xD, yD, order='lex')
t2 = time.perf_counter()
_ = sp.groebner([H2a, H2b, z * g_nd - 1], xB, yB, xC, yC, xD, yD, z, order='lex')
t3 = time.perf_counter()
print("groebner, 1 hypothese (6 var)   : %.4f s" % (t1 - t0))
print("groebner, 2 hypotheses (6 var)  : %.4f s" % (t2 - t1))
print("groebner, sature (7 var)        : %.4f s" % (t3 - t2))

groebner, 1 hypothese (6 var)   : 0.0006 s
groebner, 2 hypotheses (6 var)  : 0.0005 s
groebner, sature (7 var)        : 0.0034 s


### Lecture du résultat

La sortie chiffre chacun : `0.0006 s` pour 1 hypothèse (6 variables), `0.0005 s` pour 2 hypothèses (6 variables), `0.0034 s` pour la base saturée (7 variables) — quelques millisecondes au plus, nos idéaux sont des jouets. Sur des systèmes industriels — cinématique de robot, cryptographie, démonstration automatique à grande échelle — le même calcul peut prendre des heures ou saturer la mémoire, et le choix de l'ordre des variables devient un art.

Pour les énoncés *géométriques*, il existe une voie structurellement plus économique : la **méthode de Wu**, fondée sur la pseudo-division et les ensembles caractéristiques, qui exploite la forme triangulaire des hypothèses géométriques. C'est l'objet du 03 — et le fil rouge continuera : le même théorème du milieu, démontré par une troisième voie.

### §9 — Exercice 1 : Pythagore appartient à l'idéal

Le 01 vérifiait $P = AB^2 + AC^2 - BC^2$ par tirages ; le certificat attendu ici est une **appartenance** : montrez que $P \in \langle H \rangle$ et lisez le coefficient.

- **Étape 1.** Construisez $P$ (pensez à `sp.expand`).
- **Étape 2.** Réduisez $P$ par la base `G`.
- **Étape 3.** Rendez `(coefs, reste)` — le coefficient attendu est un entier.

In [11]:
# Exercice 1 : Pythagore appartient a l'ideal
# Etape 1 : construire P = AB^2 + AC^2 - BC^2
# Etape 2 : reduire P par la base G
# Etape 3 : retourner (coefs, reste)
P_ex1 = None        # TODO etudiant : construire P
resultat_ex1 = None  # TODO etudiant : (coefs, reste) de G.reduce(P_ex1)
print("Exercice 1 : P et le verdict d'appartenance")

Exercice 1 : P et le verdict d'appartenance


In [12]:
# Verification de l'exercice 1
if resultat_ex1 is None:
    print("Exercice 1 a completer")
else:
    coefs_e1, reste_e1 = resultat_ex1
    print("coefficient :", coefs_e1, "| reste :", reste_e1)
    print("certificat P = coefficient * H attendu :", reste_e1 == 0)

Exercice 1 a completer


### §10 — Exercice 2 : la machine dit non, encore

Le théorème « tout triangle rectangle est tel que $AB = BC$ » (côté et hypoténuse égaux) : testez l'appartenance de $AB^2 - BC^2$ à $\langle H \rangle$, puis produisez un contre-exemple **chiffré** — une figure rectangle où l'écart est visible.

- **Étape 1.** Construisez $Q = AB^2 - BC^2$.
- **Étape 2.** Réduisez par `G` et rendez le reste.
- **Étape 3.** Choisissez des coordonnées rectangle (le $3$-$4$-$5$ du §5 reste disponible) et calculez $H$ et $Q$ dessus : rendez le dictionnaire de substitution.

In [13]:
# Exercice 2 : AB = BC ?
# Etape 1 : construire Q = AB^2 - BC^2
# Etape 2 : reduire par G
# Etape 3 : contre-exemple chiffre (dictionnaire de substitution)
Q_ex2 = None          # TODO etudiant : construire Q
reste_ex2 = None      # TODO etudiant : G.reduce(Q_ex2)[1]
sub_ex2 = None        # TODO etudiant : dictionnaire {xA: ..., ..., yC: ...}
print("Exercice 2 : verdict d'appartenance + contre-exemple")

Exercice 2 : verdict d'appartenance + contre-exemple


In [14]:
# Verification de l'exercice 2
if reste_ex2 is None or sub_ex2 is None:
    print("Exercice 2 a completer")
else:
    print("reste :", reste_ex2, "| nul ?", reste_ex2 == 0)
    print("sur la figure : H =", H.subs(sub_ex2), "| Q =", Q_ex2.subs(sub_ex2))

Exercice 2 a completer


### §11 — Exercice 3 : saturations partielles

La saturation du §7 imposait $x_B^2 + y_B^2 \neq 0$ d'un seul geste. On peut viser plus fin : saturer par $x_B$ **seul** ($z \cdot x_B - 1$, c'est-à-dire $x_B \neq 0$), puis par $y_B$ seul.

- **Étape 1.** Construisez les deux bases saturées et réduisez `CONCL` par chacune : rendez les deux restes.
- **Étape 2.** Réflexion (à écrire en commentaire dans la cellule) : la première saturation démontre le théorème sur la zone $x_B \neq 0$ seulement. Vérifiez ce que valent les hypothèses et la conclusion quand $x_B = 0$ mais $y_B \neq 0$ — que force $H_{2a}$ sur $y_C$, $H_{2b}$ sur $y_D$ ? La conclusion tient-elle quand même ?
- **Étape 3.** Concluez : pourquoi la saturation complète du §7 reste-t-elle le bon geste unique ?

In [15]:
# Exercice 3 : saturations partielles z*x_B - 1 et z*y_B - 1
# Etape 1 : les deux bases et les deux restes de CONCL
# Etape 2 : que se passe-t-il sur la zone x_B = 0, y_B != 0 ?
# Etape 3 : pourquoi la saturation complete est-elle le geste general ?
reste_part1 = None  # TODO etudiant : reste de CONCL par groebner([H2a, H2b, z*xB - 1])
reste_part2 = None  # TODO etudiant : reste de CONCL par groebner([H2a, H2b, z*yB - 1])
print("Exercice 3 : deux saturations partielles")

Exercice 3 : deux saturations partielles


In [16]:
# Verification de l'exercice 3
if reste_part1 is None or reste_part2 is None:
    print("Exercice 3 a completer")
else:
    print("reste avec z*x_B - 1 :", reste_part1, "| nul ?", reste_part1 == 0)
    print("reste avec z*y_B - 1 :", reste_part2, "| nul ?", reste_part2 == 0)

Exercice 3 a completer


### §12 — La décision réelle (Z3) : un autre organe, une même question

Tout ce que le carnet a décidé jusqu'ici l'a été par bases de Gröbner : c'est une décision **algébrique**, menée dans la clôture algébrique — l'univers des **complexes**. La question « le théorème $H \Rightarrow g$ tient-il ? » y est tranchée par l'appartenance de $g$ à l'idéal $\langle H \rangle$.

Il existe un *autre* organe pour la même question, qui tranche **sur les réels** : le solveur **Z3** (théorie des corps ordonnés réels). Il ne construit pas de certificat algébrique — il *décide* la validité réelle, et, quand le théorème est faux, il **renvoie le contre-exemple lui-même** : un point réel concret, exact. Le module `real_geometry_bridge.py` de cette série l'encapsule.
Trois démonstrations suivent : le verdict réel sur le théorème du milieu (§12.1), le témoin trouvé par Z3 puis vérifié dans sympy (§12.2), et le discriminant des coniques (§12.3).

In [17]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))   # le module est dans le dossier du carnet

import real_geometry_bridge as rbg
print("organe Z3 pret :", Path(rbg.__file__).name)

organe Z3 pret : real_geometry_bridge.py


### Lecture du résultat

La sortie affiche `organe Z3 pret : real_geometry_bridge.py` : le pont vers Z3 est chargé depuis le dossier du carnet. La section 12 peut maintenant appeler `rbg.decide_real(...)` pour trancher la même question sur les **réels** — le second organe de décision, à côté du certificat algébrique des §2-§8.

#### 12.1 — Le théorème du milieu, vu des réels

Même fil rouge : $H$ (angle droit en $A$) et $C_1$ (milieu de l'hypoténuse, équidistant de $A$ et $B$). On pose la question *des réels* : existe-t-il une configuration **réelle** où $H = 0$ mais $C_1 \neq 0$ ?

Une remarque de lecture avant de lancer la décision : sur ce fil rouge, la conclusion $C_1$ **coïncide** avec l'hypothèse $H$ — le carnet l'établit plus loin en écrivant $C_1 = 1 \cdot H$. Le verdict rendu ci-dessous est donc exact, mais il ne met pas le décideur à l'épreuve : il réfute $H = 0 \wedge H \neq 0$, insatisfiable par construction. Le cas qui **discrimine** est le §12.2, où la conclusion est distincte de l'hypothèse et où la machine doit réellement chercher.


In [18]:
# Le meme theoreme, decide sur les reels par l'organe Z3
v_pos = rbg.decide_real([H], C1, [xA, yA, xB, yB, xC, yC])
print("C1 payee par H, sur les reels :", v_pos)

C1 payee par H, sur les reels : VALID — aucun contre-exemple reel


### Lecture du résultat

La sortie rend le verdict de Z3 sur les réels : `C1 payee par H, sur les reels : VALID — aucun contre-exemple reel`. Aucune figure réelle ne satisfait $H = 0$ en violant $C_1$ — exactement ce que le certificat $C_1 = 1 \cdot H$ du §2 prédisait. Le cas où le décideur doit réellement chercher un point, c'est le §12.2 ci-dessous.

#### 12.2 — Quand la machine dit non, elle *trouve* le témoin

Au §5, le contre-exemple 3-4-5 était **choisi à la main**. Ici, on demande à Z3 de le *trouver* : il renvoie un point réel qui satisfait $H$ et viole $C_{iso}$ (rectangle $\Rightarrow$ isocèle). Sans contrainte, le point le plus « facile » est dégénéré (points coïncidents) — on écarte ce cas en imposant la non-dégénérescence $B \neq A$ et $C \neq A$, le même geste que les §6-7.

In [19]:
# Non-degenerescence : B != A et C != A (sinon Z3 rend un temoin degénéré)
nd = [(xB - xA) ** 2 + (yB - yA) ** 2,
      (xC - xA) ** 2 + (yC - yA) ** 2]
v_neg = rbg.decide_real([H], Ciso, [xA, yA, xB, yB, xC, yC], assumptions=nd)
print("rectangle -> isocèle, sur les reels :", v_neg)
print("temoin trouve par Z3 :", v_neg.witness)

rectangle -> isocèle, sur les reels : INVALID — temoin reel fourni
temoin trouve par Z3 : {x_A: -1, y_A: -1, x_B: 0, y_B: -1, x_C: -1, y_C: -3}


In [20]:
# Le temoin est VERIFIE : on le substitue dans les expressions sympy
w = v_neg.witness
print("H    =", H.subs(w), "   (hypothèse satisfaite : angle droit)")
print("Ciso =", Ciso.subs(w), "   (conclusion violée : non isocele)")

H    = 0    (hypothèse satisfaite : angle droit)
Ciso = -3    (conclusion violée : non isocele)


### Lecture du résultat — le témoin, vérifié

Deux sorties se lisent ensemble. La cellule précédente a rendu `INVALID — temoin reel fourni` et exhibé le point trouvé par Z3 : `{x_A: -1, y_A: -1, x_B: 0, y_B: -1, x_C: -1, y_C: -3}`. La cellule courante le re-substitue dans les expressions sympy : `H = 0` (l'angle droit est bien satisfait) et `Ciso = -3` (la conclusion est bien violée). Le témoin n'est donc pas pris sur parole : il est **vérifié indépendamment** dans l'algèbre symbolique, et les deux organes se contrôlent l'un l'autre.

#### 12.3 — Réels vs complexes : le discriminant d'une conique

Le §7 a laissé la subtilité *réelle/complexes* en suspend. Une conique quadratique $a x^2 + b x y + c y^2 = 0$ porte son classifieur dans le **discriminant** $\Delta = b^2 - 4ac$ : $\Delta \ge 0$ elle se factorise sur $\mathbb{R}$, $\Delta < 0$ elle ne se factorise que sur $\mathbb{C}$. L'organe Z3 **vérifie** la lecture symbolique — existe-t-il un point réel non nul sur la conique ?

In [21]:
x, y = sp.symbols('x y')
for label, q in [("croisee  x^2 - y^2", x**2 - y**2),
                 ("cercle   x^2 + y^2", x**2 + y**2)]:
    d = rbg.conic_discriminant(q, x, y)
    print(f"{label:18}  delta = {int(d.delta):>3}   {d.kind:12}   "
          f"point reel non nul : {d.has_real_point}")

croisee  x^2 - y^2  delta =   4   hyperbolique   point reel non nul : True
cercle   x^2 + y^2  delta =  -4   elliptique     point reel non nul : False


### Lecture du résultat

Deux organes, une même question. La base de Gröbner **certifie** l'appartenance — un calcul algébrique valant sur les **complexes** ; Z3 **décide** la validité sur les **réels**, et il *produit* le contre-exemple au lieu de le supposer. Le témoin qu'il rend s'avère **vérifiable de l'autre côté** : substitué dans les expressions sympy, il satisfait bien l'hypothèse et viole bien la conclusion — les deux organes s'accordent, chacun dans son univers.

On y retrouve, en filigrane, le thème des §6-7 : le contre-exemple le plus « facile » pour le solveur est le *dégénéré* (points coïncidents), et c'est précisément la non-dégénérescence $B \neq A$ — rendue algébrique par la saturation de Ritt — qui l'écarte. La frontière réelle/complexes, quant à elle, n'est plus une intuition : le discriminant la **mesure**, et Z3 la **confirme**.
\n\nSur les deux coniques du §12.3, la sortie tranche en chiffres : `croisee  x^2 - y^2` donne `delta = 4`, `hyperbolique`, `point reel non nul : True` ; `cercle  x^2 + y^2` donne `delta = -4`, `elliptique`, `point reel non nul : False`. Le signe du discriminant décide bien de l'existence d'un point réel non nul : la croix se factorise sur $\mathbb{R}$, le cercle non.

## Annexe — le jumeau formel dans le lac : `Geometry.GroebnerBridge`

Ce carnet décide « prouvé par Gröbner » **par le calcul sympy** (réduction, quotients,
restes). Le lac en garde le certificat : le module
[`geometry_lean/Geometry/GroebnerBridge.lean`](geometry_lean/Geometry/GroebnerBridge.lean)
porte chaque appartenance par une identité polynomiale que Lean vérifie par `ring`, et chaque
reste non nul par un contre-exemple concret. Les énoncés ci-dessous sont recopiés depuis la
source, sans reformulation ; leur compilation reste celle de la CI du lac.

```lean
theorem fil_c1_eq_h : FilC1 = FilH
theorem fil_c1_mem_ideal : FilC1 ∈ Ideal.span {FilH}
theorem fil_cp_eq_h : FilCp = FilH
theorem fil_c3_eq_zero : FilC3 = 0
theorem fil_soundness (z : Fin 6 → ℚ) (h : eval z FilH = 0) :
    eval z FilC1 = 0 ∧ eval z FilCp = 0 ∧ eval z FilC3 = 0
theorem mem_ideal_vanishes (I : Ideal (MvPolynomial (Fin 6) ℚ))
    {f : MvPolynomial (Fin 6) ℚ} (hf : f ∈ I) (z : Fin 6 → ℚ)
    (hz : ∀ g ∈ I, eval z g = 0) : eval z f = 0
theorem fil_iso_unprovable :
    ¬ (∀ z : Fin 6 → ℚ, eval z FilH = 0 → eval z FilCiso = 0)
theorem nd_false_unprovable :
    ¬ (∀ z : Fin 6 → ℚ, eval z NdHA = 0 → eval z NdHB = 0 →
        eval z NdConcl = 0)
theorem ritt_identity : NdG * NdConcl = NdU * NdHA - NdV * NdHB
theorem ritt_mem_ideal : NdG * NdConcl ∈ Ideal.span {NdHA, NdHB}
theorem ritt_soundness (z : Fin 6 → ℚ) (ha : eval z NdHA = 0)
    (hb : eval z NdHB = 0) (hg : eval z NdG ≠ 0) : eval z NdConcl = 0
noncomputable def rabZ : MvPolynomial (Fin 7) ℚ :=
  X 6 * ((X 0)^2 + (X 1)^2) - 1
theorem rab_identity :
    (X 6 * rabU) * rabHA + -(X 6 * rabV) * rabHB + -rabConcl * rabZ
      = rabConcl
theorem rab_mem_ideal : rabConcl ∈ Ideal.span {rabHA, rabHB, rabZ}
theorem rab_soundness (z : Fin 7 → ℚ) (ha : eval z rabHA = 0)
    (hb : eval z rabHB = 0) (hz : eval z rabZ = 0) : eval z rabConcl = 0
```

### Le fil rouge : ce que « reste nul » garantit vraiment

La cellule « réduire C1 par H » rendait quotient 1, reste 0 : `fil_c1_eq_h` le démontre —
l'équidistance `MA = MB` **est** l'orthogonalité réécrite, pas une conséquence profonde.
`fil_c1_mem_ideal` en donne la lecture Mathlib (`C1 ∈ ⟨H⟩`), `fil_cp_eq_h` et
`fil_c3_eq_zero` complètent l'équidistance (les deux autres conclusions : une égale à `H`,
une identiquement nulle), et `fil_soundness` est la garantie sémantique — partout où `H`
s'annule, les trois conclusions s'annulent. `mem_ideal_vanishes` est le pont général :
l'appartenance à l'idéal entraîne l'annulation sur la variété des hypothèses.

### L'isocélité n'est pas dans l'idéal (section 5)

Le reste non nul de la cellule 5 devenait contre-exemple formel : au triangle 3-4-5
(`A = (0,0)`, `B = (3,0)`, `C = (0,4)`), l'hypothèse s'annule et `C_iso = AB² − AC²`
vaut `−7 ≠ 0`. Pas de théorème sans appartenance — l'outil de preuve sait dire non.

### Le théorème faux (section 6) : réfuté, pas seulement hors de portée

`AB ⊥ AC`, `AB ⊥ AD` « donc » `A, C, D` alignés : au point `B = A = (0,0)`,
`C = (1,0)`, `D = (0,1)`, les deux orthogonalités s'annulent (le vecteur nul est
orthogonal à tout) et le déterminant vaut `1 ≠ 0`. L'énoncé est **faux** — la leçon de la
section 6 : avant de chercher une preuve, demander ce que dit l'idéal.

### L'astuce de Ritt / Rabinowitsch (section 7), les deux faces

La saturation par `B ≠ A` rend le reste nul — `ritt_identity` en est le certificat
**explicite** (`g • concl = u • h_a − v • h_b`, exposition 1), `ritt_mem_ideal` sa lecture
d'idéal saturé, `ritt_soundness` la conclusion géométrique : le théorème redevient vrai
dès que la non-dégénérescence exclut le point singulier. L'encodage Rabinowitsch — la
variable `z` de `rabZ` est l'**inverse virtuel** qui force `g` inversible dans l'anneau
quotient — ramène la conclusion dans l'idéal augmenté (`rab_identity`, `rab_mem_ideal`),
et `rab_soundness` est le même théorème vu depuis le quotient. La section 8 (le coût
expérimental de la certitude) reste une mesure d'implémentation, hors périmètre formel.


## Conclusion

| Question | Outil | Verdict mesuré |
|---|---|---|
| $C_1$ payée par $H$ ? | division, base de Gröbner | oui — certificat $C_1 = 1 \cdot H$, reste $0$ |
| cercle circonscrit ? | appartenance de $C_p$, $C_3$ | oui — $C_p = 1 \cdot H$, $C_3 \equiv 0$ |
| rectangle $\Rightarrow$ isocèle ? | reste canonique + $3$-$4$-$5$ | **non** — reste non nul, contre-exemple |
| deux perpendiculaires $\Rightarrow$ alignement ? | Gröbner de $\langle H_{2a}, H_{2b} \rangle$ | **faux** tel quel ($B = A$), base contenant $y_B \cdot C$ |
| …avec $B \neq A$ ? | saturation $z \cdot g - 1$ | **démontré** — reste $0$ |
| *idem*, vu des réels ? | décision réelle Z3 (`real_geometry_bridge`) | **validé** — et le contre-exemple réel est **produit**, non choisi |

Ce que vous emportez :

- l'**idéal** des hypothèses et sa propriété de transfert — l'appartenance est un *certificat* ;
- la **base de Gröbner** comme décideur d'appartenance (restes canoniques, coefficients du certificat) ;
- les **non-dégénérescences** : les énoncés géométriques sont faux sans elles, et la **saturation** $z \cdot g - 1$ les rend algébriques ;
- la **décision réelle** (Z3), organe natif aux côtés du certificat algébrique : elle tranche sur $\mathbb{R}$ et *rend* le témoin — la subtilité réelle/complexes devient une **mesure** (discriminant) confirmée par le solveur ;
- le sens des deux limites : coût doublement exponentiel au pire cas, et subtilité réelle vs complexe.

L'étape suivante, *Geometry-03 — la méthode de Wu*, remplace la base de Gröbner par la **pseudo-division** et les **ensembles caractéristiques**, la voie que les systèmes réels de preuve géométrique ont préférée — appliquée au même fil rouge.

L'astuce $z \cdot g - 1$ qui a sauvé le théorème mérite un nom : c'est la **saturation** $I : g^{\infty}$. L'accrétion [Geometry-02b — Saturation et Nullstellensatz](Geometry-02b-Saturation-Nullstellensatz-Python.ipynb) la calcule pour de bon comme un idéal du plan (générateurs exhibés par élimination), dérive le pont Nullstellensatz faible→fort qui fixe le contrat, et confronte ce contrat à un témoin Z3 non dégénéré.

**Série Geometry** : [Geometry-02b — Saturation et Nullstellensatz →](Geometry-02b-Saturation-Nullstellensatz-Python.ipynb) | [Geometry-03 — La méthode de Wu →](Geometry-03-Wu-Method-Python.ipynb)

## À retenir

- **Un reste nul est un certificat, pas un verdict** : la division de `C1` par `H` rend `quotient : 1` et `reste : 0`, et la base de Gröbner rend `coefficients : [1]` — l'appartenance à l'idéal des hypothèses est **décidable**, et sa preuve tient dans un nombre.
- **Sans condition, un énoncé géométrique est faux** : le rectangle ne donne pas l'isocélie — le reste canonique de `Ciso` est non nul, et sur la figure 3-4-5 il vaut `H = 0` pour `Ciso = -7`.
- **La non-dégénérescence est indispensable, et se rend algébrique** : sans `$B \neq A$`, le contre-exemple dégénéré a `CONCL = 1` ; avec la saturation `z \cdot g - 1`, le reste redevient `0` et le théorème est démontré sous `$B \neq A$`.
- **Deux univers, deux organes** : Z3 décide sur les réels — `C1` payée par `H` est `VALID — aucun contre-exemple réel`, et le contre-exemple de l'énoncé faux est **produit** (`x_B = 0`, `y_B = -1` contre `x_C = -1`, `y_C = -3`, avec `H = 0` et `Ciso = -3` vérifiés de l'autre côté).
- **La subtilité réel/complexe se mesure** : le discriminant décide de l'existence d'un point réel non nul — `delta = 4` (hyperbolique, `True`) pour `x^2 - y^2`, `delta = -4` (elliptique, `False`) pour `x^2 + y^2`.
- **Le coût doublement exponentiel est le prix à payer** : 0.0006 s pour une hypothèse, 0.0034 s pour l'idéal saturé — rapide sur le fil rouge, mais la borne de Gröbner reste structurellement doublement exponentielle au pire cas.


## Vérifiez votre compréhension

1. La division de `C1` par `H` rend `quotient : 1` et `reste : 0`. Qu'est-ce que le `1` du quotient prouve sur la nature de l'hypothèse `H` ?
2. Le reste canonique de `Ciso` est non nul, et vaut `-7` sur la figure 3-4-5. Pourquoi le même énoncé devient-il **vrai** après saturation par `z \cdot g - 1` ?
3. Z3 rend `INVALID — temoin reel fourni` pour « rectangle ⇒ isocèle ». Qu'est-ce qui distingue le contre-exemple produit par Z3 d'un contre-exemple choisi à la main ?
4. Sur la conique `x^2 - y^2`, le discriminant vaut `4` et `point reel non nul : True` ; sur `x^2 + y^2`, il vaut `-4` et `False`. Que décide exactement le signe du discriminant ?

<details>
<summary>Réponses</summary>

1. Il prouve que `H` **divise** `C1` dans l'anneau des polynômes : la conclusion est un multiple exact de l'hypothèse, sans reste. C'est la propriété de transfert de l'idéal — l'appartenance devient un *certificat*, vérifiable en une multiplication, au lieu d'une assertion.
2. Parce que le contre-exemple du reste non nul est **dégénéré** : il correspond à `B = A`, où l'angle droit n'est pas défini. La saturation ajoute la relation `z \cdot g - 1` (avec `g` le carré de la distance `AB`), qui **interdit** ce point ; sur les configurations autorisées, le reste redevient `0`.
3. Z3 ne se contente pas de constater l'échec : il **produit** un point dont les coordonnées sont exactes, et ce point se substitue dans les expressions sympy — l'hypothèse y vaut `0` et la conclusion `-3`. Le contre-exemple est donc **vérifiable de l'autre côté** par un second langage, ce qu'un choix manuel ne garantirait pas.
4. Il décide de l'existence d'un **point réel non nul** sur la conique : `delta >= 0` signifie que la conique se factorise sur les réels (elle a des points réels), `delta < 0` qu'elle ne se factorise que sur les complexes. C'est la mesure de la frontière réel/complexe que le §12.3 cherchait.

</details>


## Pour aller plus loin

- **L'astuce de saturation, poussée jusqu'au bout** : [Geometry-02b — Saturation et Nullstellensatz](Geometry-02b-Saturation-Nullstellensatz-Python.ipynb) calcule l'idéal saturé `I : g^∞` comme un idéal du plan, exhibe ses générateurs, et exécute la dérivation du Nullstellensatz faible → fort avec les cofacteurs à l'appui.
- **La voie incrémentale** : [Geometry-03 — La méthode de Wu](Geometry-03-Wu-Method-Python.ipynb) remplace la base de Gröbner par la pseudo-division et les ensembles caractéristiques — le moteur que les systèmes de preuve géométrique ont préféré.
- **Les références** : D. Cox, J. Little, D. O'Shea, *Ideals, Varieties, and Algorithms* (4e éd.), chap. 2 §2 et chap. 4 §4 ; J. L. Rabinowitsch, *Zum Hilbertschen Nullstellensatz* (1929) — la lettre originale de l'astuce à une variable.
- **Dans ce dépôt** : la série [SMT/Z3](../../SMT/README.md) installe l'organe `real_geometry_bridge.py` utilisé ici — la décision sous contraintes comme organe natif à côté du certificat algébrique.
